<a href="https://colab.research.google.com/github/Sanjivkumar100/Emotion_Aware_Production_System/blob/main/emotion_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow keras

In [ ]:
!pip install keras

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout
from tensorflow.keras.utils import to_categorical

X_train=np.load('/content/drive/MyDrive/Emotional - Product Dataset/X_train.npy')
X_test=np.load('/content/drive/MyDrive/Emotional - Product Dataset/X_test.npy')

y_train=np.load('/content/drive/MyDrive/Emotional - Product Dataset/y_class_train.npy')
y_test=np.load('/content/drive/MyDrive/Emotional - Product Dataset/y_class_test.npy')

# Calculate num_classes based on the maximum label value present in both train and test sets.
# This ensures that to_categorical can create enough columns for all existing labels.
num_classes = int(np.max(np.concatenate((y_train, y_test)))) + 1

y_train=to_categorical(y_train,num_classes)
y_test=to_categorical(y_test,num_classes)

In [ ]:
VOCAB_SIZE=15000
MAX_LEN=120
EMBED_DIM=128


model=Sequential()

model.add(Embedding(VOCAB_SIZE,EMBED_DIM,input_length=MAX_LEN))

model.add(LSTM(128,return_sequences=False))
model.add(Dropout(0.5))



model.add(Dense(64,activation='relu'))

model.add(Dense(num_classes,activation='softmax'))

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history= model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=10,
    batch_size=64
    )

Epoch 1/10
724/724 ━━━━━━━━━━━━━━━━━━━━ 212s 288ms/step - accuracy: 0.3005 - loss: 2.7176 - val_accuracy: 0.4530 - val_loss: 2.1011
Epoch 2/10
724/724 ━━━━━━━━━━━━━━━━━━━━ 207s 286ms/step - accuracy: 0.4819 - loss: 1.9527 - val_accuracy: 0.4955 - val_loss: 1.8972
Epoch 3/10
724/724 ━━━━━━━━━━━━━━━━━━━━ 206s 285ms/step - accuracy: 0.5620 - loss: 1.5922 - val_accuracy: 0.5031 - val_loss: 1.8834
Epoch 4/10
724/724 ━━━━━━━━━━━━━━━━━━━━ 213s 294ms/step - accuracy: 0.6170 - loss: 1.3586 - val_accuracy: 0.4885 - val_loss: 1.9347
Epoch 5/10
724/724 ━━━━━━━━━━━━━━━━━━━━ 206s 285ms/step - accuracy: 0.6699 - loss: 1.1701 - val_accuracy: 0.4662 - val_loss: 2.0664
Epoch 6/10
724/724 ━━━━━━━━━━━━━━━━━━━━ 264s 288ms/step - accuracy: 0.7121 - loss: 1.0120 - val_accuracy: 0.4566 - val_loss: 2.1876
Epoch 7/10
724/724 ━━━━━━━━━━━━━━━━━━━━ 209s 289ms/step - accuracy: 0.7421 - loss: 0.8928 - val_accuracy: 0.4470 - val_loss: 2.4502
Epoch 8/10
724/724 ━━━━━━━━━━━━━━━━━━━━ 263s 290ms/step - accuracy: 0.7694 -

In [ ]:
import pickle as pkl

with open('/content/drive/MyDrive/Emotional - Product Dataset/tokenizer.pkl', 'rb') as tokenizer_file:
  tokenizer=pkl.load(tokenizer_file)
with open('/content/drive/MyDrive/Emotional - Product Dataset/label_encoder.pkl', 'rb') as label_encoder_file:
  label_encoder=pkl.load(label_encoder_file)

def predict_emotion(text,tokenizer,label_encoder,model):
  sequence=tokenizer.texts_to_sequences([text])
  padded= tf.keras.preprocessing.sequence.pad_sequences(sequence,maxlen=MAX_LEN)
  prediction=model.predict(padded)
  predicted_emotion=label_encoder.inverse_transform([np.argmax(prediction)])

  return predicted_emotion[0]

In [ ]:
def predict_emotion(text, tokenizer, label_encoder, model):

    text = text.lower()

    seq = tokenizer.texts_to_sequences([text])
    padded = tf.keras.preprocessing.sequence.pad_sequences(seq, maxlen=MAX_LEN)

    pred = model.predict(padded)

    emotion = label_encoder.inverse_transform([np.argmax(pred)])

    return emotion[0]


In [ ]:
predict_emotion(
    "I am feeling very frustrated today",
    tokenizer,
    label_encoder,
    model
)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step


'anger'

In [ ]:
model.save('emotion_model_classification.h5')